# Analysis and Visualization of Quantum Espresso normal modes

## Table of Content <a name="TOC"></a>

1. [General setups](#setups)

2. [Demonstration: Cs4SnBr6](#demo) 

3. [Normal modes of bulk Si](#bulk_si)  


### A. Learning objectives

- to read the dynamical matrices produced by QE 
- to visualize the normal modes computed by QE


### B. Use cases

- normal modes
- analysis of Quantum Espresso results
- visualization


### C. Functions

- `libra_py`
  - `packages`
    - `qe`
      - `methods`
        - [`get_QE_normal_modes`](#get_QE_normal_modes-1)
  - `normal_modes`
    - [`normal2xyz`](#normal2xyz-1)
    

### D. Classes and class members

None


## 1. General setups 
<a name="setups"></a>[Back to TOC](#TOC)

First, import all the necessary libraries:
* liblibra_core - for general data types from Libra
* libra_py - for the normal modes module
* py3Dmol - for visualization

The output of the cell below will throw a bunch of warnings, but this is not a problem nothing really serios. So just disregard them.

In [1]:
import os, sys
from liblibra_core import *
import libra_py.packages.qe.methods as QE_methods
import libra_py.normal_modes as nm
import math
import py3Dmol

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

The present tutorial comes with a set of precomputed dynamical matrices for several systems:

* **Si** - bulk silicon

* **Cs4SnBr6** - one of the inorganic perovskite systems we have been looking at recently

We will first demonstrate the capabilities using the `Cs4SnBr6_T200.dyn1` file from the Cs4SnBr6 folder


## 2. Demonstration: Cs4SnBr6
<a name="demo"></a>[Back to TOC](#TOC)

### 2.1. Reading files

For this tutorial, we only need the .dyn files produced by the Phonon code of the QE suite. 

To read the fils, we use the `get_QE_normal_modes` function of the `QE_methods` module: 
<a name="get_QE_normal_modes-1"></a>

In [2]:
help(QE_methods.get_QE_normal_modes)

Help on function get_QE_normal_modes in module libra_py.packages.qe.methods:

get_QE_normal_modes(filename, verbosity=0)
    Read a Quantum ESPRESSO phonon (.dyn) file and extract atomic structure,
    masses, normal-mode eigenvectors, and frequencies.
    
    Parameters
    ----------
    filename : str
        Name of the QE .dyn file produced by a phonon calculation.
    verbosity : int, optional
        Controls the amount of printed output:
            0 : no extra output (default)
            1 : basic information
            2 : detailed arrays
    
    Returns
    -------
    Elts : list of str, length = nat
        Atomic element symbols for each atom.
    R : np.ndarray, shape (3*nat,)
        Cartesian coordinates of all atoms (as written in the .dyn file).
    M : np.ndarray, shape (3*nat,)
        Masses associated with each Cartesian degree of freedom
        (atomic mass repeated for x, y, z).
    U : np.ndarray, shape (3*nat, 3*nat)
        Normal-mode eigenvectors. Ea

In [3]:
E, R, M, U, freqs_THz, freqs_cm1 = QE_methods.get_QE_normal_modes("Cs4SnBr6/Cs4SnBr6_T200.dyn1")

This module focuses on processing data generated with Quantum Espresso (QE)  program.
The function returns the following data:
* E - element labels of all atoms in the system
* R - coordinates of all atoms for all timesteps
* U - the matrix of the normal modes 

In [4]:
print(freqs_cm1)

[-1548.307925 -1272.289645 -1272.289645 -1265.049668 -1265.049668
  -957.342954  -943.699676  -889.600289  -871.113791  -871.113791
  -758.273878  -758.273878  -716.995057  -716.995057  -670.126667
  -661.084238  -583.650011  -583.543088  -583.543088  -505.975879
  -439.72077   -418.534569  -400.695857  -397.179243  -387.051703
  -386.036413  -386.036413  -385.399844  -382.266595  -380.128639
  -380.128639  -340.656873  -325.338624  -325.338624  -324.041841
  -322.238513  -322.238513  -314.499897  -314.499897  -312.303794
  -312.303794  -309.509028  -309.509028  -307.46851   -306.914035
  -305.718215  -305.718215  -302.769627  -302.769627  -298.741949
  -296.752676  -296.752676  -294.559269  -293.732074  -293.732074
  -291.837478  -291.837478  -290.050101  -290.050101  -286.835006
  -286.835006  -279.79516   -279.79516   -279.598399  -279.598399
  -271.773192  -271.773192  -268.559756  -264.78106   -264.78106
  -259.329626  -231.132226  -231.132226  -230.979357  -215.094747
  -207.6151

### 2.2. Visualizing the normal modes

Now, we can select which normal mode to visualize. To help us with this, we will use the `get_xyz2` function of the `normal_modes` module:
<a name="get_xyz2-1"></a>

In [5]:
help(nm.normal2xyz)

Help on function normal2xyz in module libra_py.normal_modes:

normal2xyz(elements, R, M, U, mode, mass_weighted=True, comment='Normal mode')
    Generate an XYZ-format string for visualizing a normal mode.
    
    Each atom line contains:
        X, Y, Z     : Cartesian coordinates
        UX, UY, UZ  : displacement vector components of the selected normal mode
    
    Parameters
    ----------
    elements : list of str, length = nat
        Atomic symbols (one per atom).
    R : np.ndarray, shape (3*nat,) or (nat, 3)
        Cartesian coordinates in Angstrom.
    M : np.ndarray or None, shape (3*nat,) or (nat, 3)
        Masses associated with each Cartesian degree of freedom.
        Required only if mass_weighted=True.
    U : np.ndarray, shape (3*nat, ndof)
        Normal-mode eigenvectors. Each column corresponds to one mode.
    mode : int
        Index of the normal mode to visualize (column index of U).
    mass_weighted : bool, optional
        If True, divide displacement 

This function combines the geometry (at the first step) and the computed eigenvectors to produce the string with an xyz format data, suitable for visualization with py3Dmol. 

The variables to change are:
* mode - index of the normal mode to visualize
* ampl - the amplitude of magnification factor - just for a better visualization

So, now lets see how the above data visualizes

In [6]:
mode = 100
ampl = 10
 
xyz = nm.normal2xyz(E, R, M, U, mode, mass_weighted=False, comment="Normal mode")

view = py3Dmol.view(width=800,height=400)
view.addModel(xyz,'xyz',{'vibrate': {'frames':10,'amplitude':ampl}}, viewer=(0,0))
view.setBackgroundColor('0xeeeeee')
view.setStyle({'sphere':{}})
view.animate({'loop': 'backAndForth'})
view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
print(xyz)

66
Normal mode
Cs     8.50859    0.00001    4.30457    -0.01036   -0.00080   -0.00964
Cs     5.11051   -0.00001   12.91383    -0.01036   -0.00080   -0.00964
Cs    -4.25430    7.36865    4.30457     0.00588   -0.00857   -0.00964
Cs    -2.55525    4.42584   12.91383     0.00588   -0.00857   -0.00964
Cs     2.55526    4.42583    4.30457     0.00448    0.00937   -0.00964
Cs     4.25429    7.36866   12.91383     0.00448    0.00937   -0.00964
Cs     1.69903    3.93149   10.04408    -0.03862   -0.01433   -0.00505
Cs    -1.69904    3.93150    1.43486     0.00773    0.03250   -0.00419
Cs     2.55527   11.30014   10.04408     0.03172   -0.02628   -0.00505
Cs     4.25429    8.35732    1.43486    -0.03201   -0.00956   -0.00419
Cs    -4.25429    8.35734   10.04408     0.00690    0.04061   -0.00505
Cs    -2.55525   11.30015    1.43486     0.02428   -0.02294   -0.00419
Cs     8.50859    7.86298   15.78354     0.00773    0.03250   -0.00419
Cs     5.11052    7.86300    7.17432    -0.03862   -0.01433   

## Exercise 1

How can you use the above function to make an alternative visualization of the normal modes, e.g. with the help of VMD software?  Hint: some scripting is needed.

## 3. Normal modes of bulk Si
<a name="bulk_si"></a>[Back to TOC](#TOC)

In addition to the above example, consider normal modes of Si. 

The corresponding data folder also contains the recipe for performing phonon calculations to generate the dynamical matrices using QE code.

We only need the `.dyn*` files. In particular, the `silicon.dyn1` file contains the dynamical matrix for bulk Si computed at the Gamma-point. Files like `silicon.dyn2`, `silicon.dyn3`, etc. contain the phonons at other k-points

Here, we streamline the steps. Moreover, we visualize all 6 normal modes for 2 k-points (Gamma = `*.dyn1` and another one = `*.dyn2`)

In [8]:
# Gamma-point
E1, R1, M1, U1, freqs1_THz, freqs1_cm1 = QE_methods.get_QE_normal_modes("Si/silicon.dyn1")

# another k-point
E2, R2, M2, U2, freqs2_THz, freqs2_cm1 = QE_methods.get_QE_normal_modes("Si/silicon.dyn2")

Now visualize them

In [9]:
ampl = 5

view = py3Dmol.view(width=800,height=400, viewergrid=(6,2))
for i in range(6):
    
    xyz1 = nm.normal2xyz(E1, R1, M1, U1, mode=i, mass_weighted=False, comment="Normal mode")
    xyz2 = nm.normal2xyz(E2, R2, M2, U2, mode=i, mass_weighted=False, comment="Normal mode")
    
    view.addModel(xyz1,'xyz',{'vibrate': {'frames':10,'amplitude':ampl}}, viewer=(i,0))
    view.addModel(xyz2,'xyz',{'vibrate': {'frames':10,'amplitude':ampl}}, viewer=(i,1))

view.setBackgroundColor('0xeeeeee')
view.setStyle({'sphere':{}})
view.animate({'loop': 'backAndForth'})
view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.